In [ ]:
import numpy as np
from sklearn.cluster import KMeans

# 1. Load data
era5_data = np.load("era5_cnn_512_embeddings.npy", allow_pickle=True).item()
events_data = np.load('events.npy', allow_pickle=True)

# 2. Extract Matrices
era5_ids = list(era5_data.keys())
era5_matrix = np.array([era5_data[eid] for eid in era5_ids])
kg_ids_raw = events_data[:, 0]
kg_matrix = np.stack(events_data[:, 1])

# 3. K-Means Priors (P(Xj))
num_clusters = 20

def compute_prior_map(matrix, ids, clean_func=lambda x: x):
    kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10).fit(matrix)
    # Distance from centroid = Distinctiveness
    distances = np.linalg.norm(matrix - kmeans.cluster_centers_[kmeans.labels_], axis=1)
    # Scale to [0.5, 1.5]
    priors = 0.5 + (distances - distances.min()) / (distances.max() - distances.min() + 1e-8)
    return {clean_func(ids[i]): priors[i] for i in range(len(priors))}

# Mapping (ensure consistency)
v_prior_map = compute_prior_map(era5_matrix, era5_ids)
d_prior_map = compute_prior_map(kg_matrix, kg_ids_raw, clean_func=lambda x: str(x).replace("name=", "").replace("_", "-").strip())

print("Priors ready.")

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import seaborn as sns

# Constants
num_clusters = 20

# --- CNN (Atmospheric) Clustering ---
kmeans_v = KMeans(n_clusters=num_clusters, random_state=42, n_init=10).fit(era5_matrix)
v_labels = kmeans_v.labels_
v_centers = kmeans_v.cluster_centers_
v_distances = np.linalg.norm(era5_matrix - v_centers[v_labels], axis=1)

# --- KG (Contextual) Clustering ---
kmeans_d = KMeans(n_clusters=num_clusters, random_state=42, n_init=10).fit(kg_matrix)
d_labels = kmeans_d.labels_
d_centers = kmeans_d.cluster_centers_
d_distances = np.linalg.norm(kg_matrix - d_centers[d_labels], axis=1)

print("Clustering complete. Models 'kmeans_v' and 'kmeans_d' are now defined.")

Engine 1 (CNN-Embeddings)

In [ ]:
import os
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

base_results_dir = "final_results"
#pangu_path = "pangu_cnn_512_trained_v2.npy"
pangu_path = "pangu_cnn_512_embeddings.npy"
pangu_data = np.load(pangu_path, allow_pickle=True).item()
p_power = 10

print(f"Processing CNN Engine for {len(pangu_data)} events...")

for pangu_id, pangu_vec in tqdm(pangu_data.items()):
    event_folder = os.path.join(base_results_dir, pangu_id)
    os.makedirs(event_folder, exist_ok=True)
    
    # 1. Likelihood (P(C|Xj))
    query_vec = pangu_vec.reshape(1, -1)
    similarities = cosine_similarity(query_vec, era5_matrix).flatten()
    likelihood = np.power(np.maximum(similarities, 0), p_power)
    
    # 2. Prior (P(Xj))
    priors = np.array([v_prior_map.get(eid, 0.5) for eid in era5_ids])
    
    # 3. Combine
    df = pd.DataFrame({
        'era5_event_id': era5_ids,
        'cnn_similarity_score': similarities,
        'cnn_likelihood': likelihood,
        'cnn_prior': priors,
        'cnn_bayesian_score': likelihood * priors
    }).sort_values('cnn_bayesian_score', ascending=False)
    
    df.to_csv(os.path.join(event_folder, "cnn_bayesian_results.csv"), index=False)

Engine 2 (KG-Embeddings)

In [ ]:
def normalize_id(text):
    return str(text).replace("name=", "").replace("_", "-").strip()

# Mapping for the search query
kg_lookup = {normalize_id(label): i for i, label in enumerate(kg_ids_raw)}
# Pre-clean the IDs for the CSV output
clean_kg_ids = [normalize_id(eid) for eid in kg_ids_raw]

print(f"Processing Expert Engine...")

pangu_folders = [f for f in os.listdir(base_results_dir) if os.path.isdir(os.path.join(base_results_dir, f))]

for folder_name in tqdm(pangu_folders):
    norm_name = normalize_id(folder_name)
    
    if norm_name in kg_lookup:
        target_idx = kg_lookup[norm_name]
        query_vec = kg_matrix[target_idx].reshape(1, -1)
        
        # 1. Likelihood (P(C|Xj))
        similarities = cosine_similarity(query_vec, kg_matrix).flatten()
        likelihood = np.power(np.maximum(similarities, 0), p_power)
        
        # 2. Prior (P(Xj))
        priors = np.array([d_prior_map.get(eid, 0.5) for eid in clean_kg_ids])
        
        # 3. Combine
        df_kg = pd.DataFrame({
            'kg_event_id': clean_kg_ids,
            'original_id': kg_ids_raw,
            'kg_similarity_score': similarities,
            'kg_likelihood': likelihood,
            'kg_prior': priors,
            'kg_bayesian_score': likelihood * priors
        }).sort_values('kg_bayesian_score', ascending=False)
        
        df_kg.to_csv(os.path.join(base_results_dir, folder_name, "graph_bayesian_results.csv"), index=False)

Fusion (Bayesian Framework)

In [ ]:
W_custom = 1.0 
k_custom = 5

for folder in tqdm(os.listdir(base_results_dir), desc="Final Fusion"):
    cnn_p = os.path.join(base_results_dir, folder, "cnn_bayesian_results.csv")
    kg_p = os.path.join(base_results_dir, folder, "graph_bayesian_results.csv")
    
    if os.path.exists(cnn_p) and os.path.exists(kg_p):
        df_cnn = pd.read_csv(cnn_p)
        df_kg = pd.read_csv(kg_p)
        
        df_cnn['match_id'] = df_cnn['era5_event_id'].apply(normalize_id)
        df_kg['match_id'] = df_kg['kg_event_id'].apply(normalize_id)
        
        df_fused = pd.merge(
            df_cnn[['match_id', 'era5_event_id', 'cnn_bayesian_score']],
            df_kg[['match_id', 'kg_bayesian_score']],
            on='match_id'
        )
        
        # Get Rank from Expert Engine
        df_fused['kg_rank'] = df_fused['kg_bayesian_score'].rank(ascending=False)
        
        # Bayesian Fusion Formula
        df_fused['attribution_weight'] = df_fused['cnn_bayesian_score'] * (
            1.0 + (W_custom / (k_custom + df_fused['kg_rank']))
        )
        
        df_fused.sort_values('attribution_weight', ascending=False).to_csv(
            os.path.join(base_results_dir, folder, "final_bayesian_attribution.csv"), index=False
        )

Compare Results

In [ ]:
import os
import pandas as pd
import numpy as np
from scipy.stats import kendalltau

base_results_dir = "final_results"
stats = []

print("Analyzing Framework Impact across all events...")

for folder in os.listdir(base_results_dir):
    fused_path = os.path.join(base_results_dir, folder, "final_bayesian_attribution.csv")
    cnn_raw_path = os.path.join(base_results_dir, folder, "cnn_bayesian_results.csv") # contains raw_sim
    
    if os.path.exists(fused_path) and os.path.exists(cnn_raw_path):
        df_fused = pd.read_csv(fused_path)
        df_cnn = pd.read_csv(cnn_raw_path)
        
        # 1. Compare Top 25 Lists
        raw_top_25 = set(df_cnn.sort_values('cnn_similarity_score', ascending=False).head(25)['era5_event_id'])
        fused_top_25 = set(df_fused.head(25)['era5_event_id'])
        
        # Intersection: How many of the same events stayed in the top 25?
        common_count = len(raw_top_25.intersection(fused_top_25))
        
        # 2. Check Top 1 Identity
        raw_top_1 = df_cnn.sort_values('cnn_similarity_score', ascending=False).iloc[0]['era5_event_id']
        fused_top_1 = df_fused.iloc[0]['era5_event_id']
        top_1_changed = (raw_top_1 != fused_top_1)
        
        # 3. Calculate the Decisiveness Gap (Rank 1 vs Rank 2)
        # Raw Gap
        raw_scores = df_cnn.sort_values('cnn_similarity_score', ascending=False)['cnn_similarity_score'].values
        raw_gap = raw_scores[0] - raw_scores[1]
        
        # Fused Gap (Normalized so we can compare to raw)
        fused_scores = df_fused['attribution_weight'].values
        fused_gap = (fused_scores[0] - fused_scores[1]) / fused_scores[0] # Percentage gap
        
        stats.append({
            'folder': folder,
            'common_top_25': common_count,
            'top_1_changed': top_1_changed,
            'raw_gap': raw_gap,
            'fused_relative_gap': fused_gap
        })

# --- Convert to Summary DataFrame ---
df_stats = pd.DataFrame(stats)

print(f"\n{'='*40}")
print(f"GLOBAL FRAMEWORK EVALUATION")
print(f"{'='*40}")
print(f"Average Events Retained in Top 25: {df_stats['common_top_25'].mean():.2f} / 25")
print(f"Top 1 Change Frequency: {df_stats['top_1_changed'].mean()*100:.1f}% of events")
print(f"Average Raw Gap (Sim): {df_stats['raw_gap'].mean():.4f}")
print(f"Average Bayesian Confidence Gap: {df_stats['fused_relative_gap'].mean()*100:.2f}%")

In [ ]:
import os
import pandas as pd
import numpy as np

# --- Configuration ---
RESULTS_DIR = 'final_results' 
MERGED_DATA_CSV = 'merged_events_countries.csv'

def parse_to_set(val):
    """Safely converts a comma-separated string into a Python set."""
    if pd.isna(val) or str(val).strip() == '' or str(val).strip().lower() == 'nan':
        return set()
    return set(item.strip() for item in str(val).split(',') if item.strip())

def compare_sets(query_set, match_set):
    """Returns the number of common items and the Jaccard similarity percentage."""
    # If either country is missing data, we can't compare them. Return NaN.
    if not query_set or not match_set:
        return 0, np.nan 
    
    common = query_set.intersection(match_set)
    union = query_set.union(match_set)
    jaccard_pct = (len(common) / len(union)) * 100
    return len(common), round(jaccard_pct, 2)

print("Loading merged country/environmental data...")
reference_df = pd.read_csv(MERGED_DATA_CSV)

# 1. Build a fast lookup dictionary from our merged CSV
event_info = {}
for _, row in reference_df.iterrows():
    event_info[row['Event_id']] = {
        'Country': row['Country'],
        'Species': parse_to_set(row['All Species']),
        'Landcover': parse_to_set(row['All Landcover Types'])
    }

# 2. Iterate through the results folders
evaluation_results = []
subfolders = [f for f in os.listdir(RESULTS_DIR) if os.path.isdir(os.path.join(RESULTS_DIR, f))]

print(f"Found {len(subfolders)} query event folders. Processing Top 10 matches...")

for folder in subfolders:
    query_event_id = folder.replace('_', '-')
    
    if query_event_id not in event_info:
        continue
        
    query_data = event_info[query_event_id]
    
    csv_path = os.path.join(RESULTS_DIR, folder, 'final_bayesian_attribution.csv')
    if not os.path.exists(csv_path):
        continue
        
    rankings = pd.read_csv(csv_path)
    
    # Grab the Top 10 instead of Top 5
    top_10_matches = rankings.sort_values(by='attribution_weight', ascending=False).head(100)
    
    for rank, row in enumerate(top_10_matches.iterrows(), 1):
        _, row_data = row
        match_id = row_data['match_id']
        
        if match_id not in event_info:
            continue
            
        match_data = event_info[match_id]
        
        is_same_country = (query_data['Country'] == match_data['Country'])
        
        # 3. Calculate Ecological Overlaps
        common_species, species_sim = compare_sets(query_data['Species'], match_data['Species'])
        common_lc, lc_sim = compare_sets(query_data['Landcover'], match_data['Landcover'])
        
        evaluation_results.append({
            'Query_Event': query_event_id,
            'Query_Country': query_data['Country'],
            'Rank': rank,
            'Matched_Event': match_id,
            'Matched_Country': match_data['Country'],
            'Exact_Country_Match': is_same_country,
            'Shared_Landcovers': common_lc,
            'Landcover_Overlap_%': lc_sim,
            'Shared_Species': common_species,
            'Species_Overlap_%': species_sim,
            'Attribution_Weight': round(row_data['attribution_weight'], 5)
        })

# 4. Save the full results
eval_df = pd.DataFrame(evaluation_results)
eval_df.to_csv('ecological_evaluation_top100.csv', index=False)

# 5. Display ONLY the interesting cross-country matches!
cross_country_df = eval_df[eval_df['Exact_Country_Match'] == False].copy()

print("\n--- CROSS-COUNTRY MATCHES (ECOLOGICAL SIMILARITY) ---")
columns_to_show = ['Query_Country', 'Rank', 'Matched_Country', 'Shared_Landcovers', 'Landcover_Overlap_%', 'Shared_Species', 'Species_Overlap_%']

if cross_country_df.empty:
    print("No cross-country matches found in the top 10.")
else:
    # We use .to_string() here to avoid the tabulate error you hit earlier!
    print(cross_country_df[columns_to_show].head(15).to_string(index=False))

print("\nSaved full dataset to 'ecological_evaluation_top10.csv'")

In [ ]:
import pandas as pd

# 1. Load the evaluation dataset
# Ensure the CSV file is in the same directory as this script
df = pd.read_csv('ecological_evaluation_top100.csv')

# 2. Calculate Total Matches
total_matches = len(df)

# 3. Separate the data into Exact Matches and Cross-Country Matches
exact_matches_df = df[df['Exact_Country_Match'] == True]
cross_country_df = df[df['Exact_Country_Match'] == False]

# Count how many fall into each category
exact_count = len(exact_matches_df)
cross_count = len(cross_country_df)

# Calculate the percentages
exact_pct = (exact_count / total_matches) * 100
cross_pct = (cross_count / total_matches) * 100

# 4. Calculate Ecological Similarity (Average Landcover Overlap)
# We calculate the mean of the overlap, using .dropna() to ignore any rows 
# where a country happened to have missing environmental data
avg_landcover_overlap = cross_country_df['Landcover_Overlap_%'].dropna().mean()

# --- Print the Results ---
print(f"Total Top-10 Matches Evaluated: {total_matches}\n")

print(f"Exact Country Matches: {exact_pct:.1f}% of the time, your model retrieves an event from the exact same country. ({exact_count} matches)")

print(f"\nCross-Country Matches: {cross_pct:.1f}% ({cross_count} matches) retrieve an event from a different country.")

print(f"\nEcological Similarity: When the model jumps to a different country, the average Landcover Overlap is {avg_landcover_overlap:.1f}%.")

In [ ]:
import os
import pandas as pd
import re
import collections
import math

# --- Configuration ---
RESULTS_DIR = 'final_results' 

# 1. Parse the original raw Neo4j export to get Categories
print("Step 1: Extracting Taxonomic Categories...")
raw_bio_df = pd.read_csv('export_bio.csv')

def extract_categories(s):
    """Extracts the 'category' property from the Neo4j string."""
    if pd.isna(s) or s == '[]':
        return []
    matches = re.findall(r'category:\s*([^,]+),', str(s))
    return [match.strip() for match in matches]

def extract_country_name(s):
    if pd.isna(s): return 'Unknown'
    match = re.search(r'name:\s*([^,]+),', str(s))
    return match.group(1).strip() if match else 'Unknown'

def build_taxonomic_profile(category_list):
    """Converts a list of categories into a percentage dictionary."""
    if not category_list:
        return {}
    counts = collections.Counter(category_list)
    total = sum(counts.values())
    return {category: count / total for category, count in counts.items()}

# Build profiles for each country
country_profiles = {}
for _, row in raw_bio_df.iterrows():
    country_name = extract_country_name(row['region_metadata'])
    categories = extract_categories(row['biodiversity_data'])
    country_profiles[country_name] = build_taxonomic_profile(categories)

# 2. Map Events to Countries (from your export_country.csv)
print("Step 2: Mapping Events to Countries...")
events_df = pd.read_csv('export_country.csv')
event_to_country = dict(zip(events_df['Event_id'], events_df['Country']))

# 3. Define Cosine Similarity Math
def calculate_cosine_similarity(profile1, profile2):
    """Calculates cosine similarity between two percentage dictionaries."""
    if not profile1 or not profile2:
        return float('nan') # Return NaN if a country is missing data
    
    # Get all unique categories across both countries
    all_categories = set(profile1.keys()).union(set(profile2.keys()))
    
    # Build aligned vectors
    vec1 = [profile1.get(cat, 0.0) for cat in all_categories]
    vec2 = [profile2.get(cat, 0.0) for cat in all_categories]
    
    # Cosine Similarity Formula: dot_product / (norm(vec1) * norm(vec2))
    dot_product = sum(a * b for a, b in zip(vec1, vec2))
    norm1 = math.sqrt(sum(a * a for a in vec1))
    norm2 = math.sqrt(sum(b * b for b in vec2))
    
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return dot_product / (norm1 * norm2)

# 4. Evaluate the CNN Rankings
print("Step 3: Evaluating CNN Matches...")
evaluation_results = []
subfolders = [f for f in os.listdir(RESULTS_DIR) if os.path.isdir(os.path.join(RESULTS_DIR, f))]

for folder in subfolders:
    query_event_id = folder.replace('_', '-')
    query_country = event_to_country.get(query_event_id)
    
    if not query_country or query_country not in country_profiles: continue
    query_profile = country_profiles[query_country]
    
    csv_path = os.path.join(RESULTS_DIR, folder, 'final_bayesian_attribution.csv')
    if not os.path.exists(csv_path): continue
        
    rankings = pd.read_csv(csv_path)
    top_10 = rankings.sort_values(by='cnn_bayesian_score', ascending=False).head(10)
    
    for rank, row in enumerate(top_10.iterrows(), 1):
        _, row_data = row
        match_id = str(row_data['era5_event_id']).replace('_', '-')
        match_country = event_to_country.get(match_id)
        
        if not match_country or match_country not in country_profiles: continue
        match_profile = country_profiles[match_country]
        
        is_same = (query_country == match_country)
        taxonomic_sim = calculate_cosine_similarity(query_profile, match_profile)
        
        evaluation_results.append({
            'Query_Country': query_country,
            'Rank': rank,
            'Matched_Country': match_country,
            'Exact_Country_Match': is_same,
            'Taxonomic_Similarity_%': round(taxonomic_sim * 100, 2) if not pd.isna(taxonomic_sim) else 'No Data',
            'CNN_Score': round(row_data['cnn_bayesian_score'], 5)
        })

eval_df = pd.DataFrame(evaluation_results)

# 5. Print Results!
cross_country_df = eval_df[eval_df['Exact_Country_Match'] == False].copy()
# Filter out "No Data" rows to calculate the true average
valid_cross_country = cross_country_df[cross_country_df['Taxonomic_Similarity_%'] != 'No Data']

avg_taxonomic = valid_cross_country['Taxonomic_Similarity_%'].astype(float).mean()

print(f"\n--- TAXONOMIC EVALUATION RESULTS ---")
print(f"Average Taxonomic Similarity (Cross-Country): {avg_taxonomic:.1f}%")
print("\nTop 15 Cross-Country Matches by Taxonomic Similarity:")
print(valid_cross_country.sort_values(by='Taxonomic_Similarity_%', ascending=False).head(15).to_string(index=False))

eval_df.to_csv('cnn_taxonomic_evaluation.csv', index=False)